In [1]:
import tensorflow as tf
import numpy as np
import random
import gc
import os
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy.io import loadmat
import keras_hub
from keras_hub.layers import TransformerEncoder

from collections import defaultdict

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Layer,
    Dense,
    Dropout,
    Flatten,
    Reshape,
    Permute,
    Conv2D,
    BatchNormalization,
    LayerNormalization,
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Loss
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras import layers

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import ast
import pandas as pd


2026-04-23 18:42:00.867320: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776969721.092709      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776969721.160880      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776969721.695347      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776969721.695406      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776969721.695411      23 computation_placer.cc:177] computation placer alr

In [2]:
class EEGDataset_ADHD_TF:
    """
    Subject-level dataset (one .mat file = one subject)
    Each item: (fname, eeg_epochs_np, label_int)
      eeg_epochs_np: (E, C, T)
    """

    def __init__(self, adhd_dir, control_dir,
                 lowcut=0.5, highcut=60.0, notch=50.0,
                 window=2.0, overlap=0.5,
                 default_fs=128):

        self.samples = []
        self.lowcut = float(lowcut)
        self.highcut = float(highcut)
        self.notch = float(notch)
        self.window = float(window)
        self.overlap = float(overlap)
        self.default_fs = int(default_fs)

        self._process_folder(adhd_dir, label=1)
        self._process_folder(control_dir, label=0)

        if len(self.samples) == 0:
            raise ValueError("No subjects loaded. Check folders and file contents.")

    def _process_folder(self, folder, label):
        if not os.path.isdir(folder):
            raise ValueError(f"Directory does not exist: {folder}")

        for fname in sorted(os.listdir(folder)):
            if not fname.lower().endswith(".mat"):
                continue
            mat_path = os.path.join(folder, fname)
            eeg = self._process_mat(mat_path)
            if eeg is not None:
                self.samples.append((fname, eeg.astype(np.float32), int(label)))

    def _process_mat(self, file_path):
        mat = loadmat(file_path)

        # Variable name is usually the same as filename (e.g., v10p.mat -> "v10p")
        key = os.path.splitext(os.path.basename(file_path))[0]
        if key not in mat:
            # fallback: first 2D array
            key = None
            for k, v in mat.items():
                if isinstance(v, np.ndarray) and v.ndim == 2 and v.size > 0:
                    key = k
                    break
            if key is None:
                return None

        data = np.asarray(mat[key], dtype=np.float64)  # often (T, C)

        # Find fs if present
        fs = None
        for k in mat.keys():
            kl = k.lower()
            if ("fs" in kl) or ("freq" in kl) or ("sampling" in kl) or ("sfreq" in kl):
                try:
                    fs = int(np.squeeze(mat[k]))
                    break
                except Exception:
                    fs = None
        if fs is None or fs <= 0:
            fs = self.default_fs

        # Ensure (C, T) for MNE
        if data.ndim != 2:
            return None

        if data.shape[1] <= 256 and data.shape[0] > data.shape[1]:
            data = data.T  # (C, T)

        n_ch, n_times = data.shape
        min_samples = int(np.ceil(self.window * fs))
        if n_times < min_samples:
            return None

        ch_names = [f"Ch{i+1}" for i in range(n_ch)]
        info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=["eeg"] * n_ch)
        raw = mne.io.RawArray(data, info, verbose=False)

        raw.set_eeg_reference("average", verbose=False)
        raw.notch_filter(freqs=[self.notch], method="iir", verbose=False)
        raw.filter(self.lowcut, self.highcut, method="iir", verbose=False)

        step = self.window * (1.0 - self.overlap)
        if step <= 0:
            raise ValueError("overlap must be < 1.0 so that step > 0")

        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=self.window,
            overlap=self.window - step,
            preload=True,
            verbose=False
        )

        eeg_data = epochs.get_data()  # (E, C, T)
        if eeg_data.size == 0:
            return None

        return eeg_data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [3]:
# Build epoch-level arrays from subject-level dataset
def build_epoch_arrays(dataset_obj):
    X_list, y_list, g_list = [], [], []

    for i in range(len(dataset_obj)):
        name, eeg_epochs, label = dataset_obj[i]  # (E,C,T)
        E = int(eeg_epochs.shape[0])

        X_list.append(eeg_epochs)
        y_list.append(np.full((E,), int(label), dtype=np.int32))
        g_list.append(np.full((E,), str(name), dtype=object))

    X = np.concatenate(X_list, axis=0).astype(np.float32)   # (N,C,T)
    y = np.concatenate(y_list, axis=0).astype(np.int32)     # (N,)
    groups = np.concatenate(g_list, axis=0)                 # (N,) subject-name per epoch

    return X, y, groups


# Subject-level labels derived from epoch-level groups/y
def build_subject_table_from_epochs(groups_epoch, y_epoch):
    subj_names = np.unique(groups_epoch.astype(str))
    subj_labels = []

    for s in subj_names:
        ys = np.unique(y_epoch[groups_epoch.astype(str) == s])
        if len(ys) != 1:
            raise ValueError(f"Subject {s} has multiple labels across epochs: {ys}")
        subj_labels.append(int(ys[0]))

    return subj_names, np.array(subj_labels, dtype=np.int32)


def epoch_indices_from_subjects(groups_epoch, subject_names_subset):
    subject_names_subset = np.array(subject_names_subset).astype(str)
    return np.where(np.isin(groups_epoch.astype(str), subject_names_subset))[0]


# tf.data builders
def make_ds_from_indices(X, y, groups, idxs, training, with_groups, batch_size, seed):
    x = X[idxs]
    yy = y[idxs]

    if with_groups:
        gg = groups[idxs].astype(str)
        ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


# ============================================================
# COMPATIBILIDAD TGARNET: salida dict / sigmoid / softmax
# ============================================================
def extract_positive_prob(model_output):
    """
    Compatible con:
      - dict output de TGARNet: {"out_activation": ..., "kernel_entropy": ...}
      - sigmoid output: (B, 1) o (B,)
      - softmax output: (B, 2)

    Retorna:
      prob_pos: (B,)
    """
    if isinstance(model_output, dict):
        if "out_activation" not in model_output:
            raise ValueError(
                f"Dict output sin 'out_activation'. Keys: {list(model_output.keys())}"
            )
        out = model_output["out_activation"]
    else:
        out = model_output

    if tf.is_tensor(out):
        out = out.numpy()

    out = np.asarray(out)

    if out.ndim == 1:
        return out.astype(np.float32)

    if out.ndim == 2:
        if out.shape[1] == 1:
            return out[:, 0].astype(np.float32)
        if out.shape[1] == 2:
            return out[:, 1].astype(np.float32)

    raise ValueError(f"Unsupported model output shape: {out.shape}")

class ValSubjectBalancedAccuracy(tf.keras.callbacks.Callback):
    """
    Validation balanced accuracy at SUBJECT level using majority vote.

    Expects val_ds to yield:
      (x, y, group_name)
    where:
      x : EEG epoch batch
      y : epoch labels
      group_name : subject id per epoch
    """
    def __init__(self, val_ds_with_groups, threshold=0.5):
        super().__init__()
        self.val_ds = val_ds_with_groups
        self.threshold = float(threshold)
        self.best = -np.inf
        self.last = None

    def on_epoch_end(self, epoch, logs=None):
        subj_probs = {}
        subj_true = {}

        for xb, yb, gb in self.val_ds:
            raw_pred = self.model(xb, training=False).numpy()
            prob_pos = extract_positive_prob(raw_pred)

            y_np = yb.numpy().astype(int)
            g_np = gb.numpy()

            for p, y_true_i, g_i in zip(prob_pos, y_np, g_np):
                g_str = g_i.decode("utf-8") if isinstance(g_i, bytes) else str(g_i)

                if g_str not in subj_probs:
                    subj_probs[g_str] = []
                    subj_true[g_str] = int(y_true_i)

                subj_probs[g_str].append(float(p))

        y_true_subject = []
        y_pred_subject = []

        for subj in sorted(subj_probs.keys()):
            probs = np.array(subj_probs[subj], dtype=np.float32)
            epoch_preds = (probs >= self.threshold).astype(int)

            # Majority vote across epochs
            pred_subj = int(np.round(epoch_preds.mean()))
            true_subj = int(subj_true[subj])

            y_true_subject.append(true_subj)
            y_pred_subject.append(pred_subj)

        bacc = balanced_accuracy_score(y_true_subject, y_pred_subject)
        self.last = float(bacc)
        self.best = max(self.best, self.last)

        if logs is not None:
            logs["val_subject_bacc"] = self.last

        print(f" - val_subject_bacc: {self.last:.4f}", end="")


# Plotting functions
def plot_fold_training_curve(history, fold_id, save_path, metrics=("loss", "acc", "auc")):
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes = [axes]
    fig.suptitle(f"Fold {fold_id + 1} — Training Curves (Final Retrain)", fontsize=13)
    epochs_range = range(1, len(history["loss"]) + 1)
    for col, m in enumerate(metrics):
        ax = axes[col]
        if m in history:
            ax.plot(epochs_range, history[m], label=f"train_{m}", linewidth=1.2)
        val_key = f"val_{m}"
        if val_key in history:
            ax.plot(epochs_range, history[val_key], label=val_key, linewidth=1.2, linestyle="--")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


# Fold-level training curves for all folds in a grid + overlay of all folds per metric
def plot_all_training_curves(fold_histories, save_dir, metrics=("loss", "acc", "auc")):
    n_folds = len(fold_histories)
    n_metrics = len(metrics)

    fig, axes = plt.subplots(n_folds, n_metrics, figsize=(5 * n_metrics, 3.5 * n_folds), squeeze=False)
    fig.suptitle("Training Curves per Fold (Final Retrain)", fontsize=14, y=1.01)
    for row, fid in enumerate(sorted(fold_histories.keys())):
        hist = fold_histories[fid]
        epochs_range = range(1, len(hist["loss"]) + 1)
        for col, m in enumerate(metrics):
            ax = axes[row, col]
            if m in hist:
                ax.plot(epochs_range, hist[m], label=f"train_{m}", linewidth=1.0)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=val_key, linewidth=1.0, linestyle="--")
            ax.set_title(f"Fold {fid + 1} — {m}", fontsize=9)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(m)
            ax.legend(fontsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, "training_curves_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, axes2 = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes2 = [axes2]
    fig2.suptitle("Training Curves Overlay (All Folds)", fontsize=14)
    for col, m in enumerate(metrics):
        ax = axes2[col]
        for fid in sorted(fold_histories.keys()):
            hist = fold_histories[fid]
            epochs_range = range(1, len(hist["loss"]) + 1)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
            elif m in hist:
                ax.plot(epochs_range, hist[m], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
        ax.set_title(f"All Folds — {m}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "training_curves_overlay.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")


# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_fold_confusion_matrix(cm, fold_id, save_path, class_names=("Control", "ADHD")):
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Fold {fold_id + 1} — Confusion Matrix (Subject-Level)", fontsize=11)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_all_confusion_matrices(fold_cms, super_cm, save_dir, class_names=("Control", "ADHD")):
    n_folds = len(fold_cms)
    ncols = min(n_folds, 5)
    nrows = int(np.ceil(n_folds / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle("Confusion Matrices per Fold (Subject-Level)", fontsize=14, y=1.01)
    for i, fid in enumerate(sorted(fold_cms.keys())):
        row, col = divmod(i, ncols)
        ax = axes[row][col]
        disp = ConfusionMatrixDisplay(fold_cms[fid], display_labels=class_names)
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(f"Fold {fid + 1}", fontsize=10)
    for j in range(i + 1, nrows * ncols):
        row, col = divmod(j, ncols)
        axes[row][col].axis("off")
    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrices_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 5))
    disp2 = ConfusionMatrixDisplay(super_cm, display_labels=class_names)
    disp2.plot(ax=ax2, cmap="Oranges", colorbar=False, values_format="d")
    ax2.set_title("Accumulated Confusion Matrix (All Folds)", fontsize=12)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "confusion_matrix_accumulated.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

# Model


In [4]:
@tf.keras.utils.register_keras_serializable()
class GaussianKernelLayer(Layer):
    def __init__(self, **kwargs):
        super(GaussianKernelLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        super(GaussianKernelLayer, self).build(input_shape)

    def call(self, inputs):
        """
        inputs: [x, sigma]
        x shape: (N, C, T, F)
        sigma: escalar o tensor compatible
        """
        x, sigma = inputs

        N = tf.shape(x)[0]
        C = tf.shape(x)[1]
        T = tf.shape(x)[2]
        F = tf.shape(x)[3]

        # (N, C, T, F) -> (N, F, C, T)
        x = tf.transpose(x, perm=(0, 3, 1, 2))

        # (N, F, C, T) -> (N*F, C, T)
        x_reshaped = tf.reshape(x, (N * F, C, T))

        # Distancias cuadradas por pares entre canales
        squared_differences = (
            tf.expand_dims(x_reshaped, axis=2) - tf.expand_dims(x_reshaped, axis=1)
        )  # (N*F, C, C, T)
        squared_differences = tf.square(squared_differences)
        pairwise_distances_squared = tf.reduce_sum(squared_differences, axis=-1)  # (N*F, C, C)

        # (N*F, C, C) -> (N, F, C, C) -> (N, C, C, F)
        pairwise_distances_squared = tf.reshape(pairwise_distances_squared, (N, F, C, C))
        pairwise_distances_squared = tf.transpose(pairwise_distances_squared, perm=(0, 2, 3, 1))

        gaussian_kernel = tf.exp(-pairwise_distances_squared / (2.0 * tf.square(sigma)))
        return gaussian_kernel
@tf.keras.utils.register_keras_serializable()
class RenyiEntropyLayer(tf.keras.layers.Layer):
    def __init__(self, alpha=2, eps=1e-8, normalize_by_logc=False, **kwargs):
        super(RenyiEntropyLayer, self).__init__(**kwargs)
        self.alpha = alpha
        self.eps = eps
        self.normalize_by_logc = normalize_by_logc

    def call(self, K):
        """
        K: tensor de forma (B, F, C, C)
           En tu caso con un solo kernel: (B, 1, C, C)

        retorna:
           H: tensor de forma (B, F)
        """
        K = tf.cast(K, tf.float32)

        # Normalización por traza
        trace = tf.linalg.trace(K)                          # (B, F)
        trace = tf.maximum(trace, self.eps)
        A = K / trace[..., None, None]                     # (B, F, C, C)

        if self.alpha == 2:
            AAT = tf.matmul(A, A, transpose_b=True)        # (B, F, C, C)
            H = -tf.math.log(tf.maximum(tf.linalg.trace(AAT), self.eps))  # (B, F)
        else:
            eigvals, _ = tf.linalg.eigh(A)                 # (B, F, C)
            eigvals = tf.maximum(tf.math.real(eigvals), self.eps)
            H = tf.math.log(tf.reduce_sum(tf.pow(eigvals, self.alpha), axis=-1)) / (1.0 - self.alpha)

        if self.normalize_by_logc:
            C = tf.cast(tf.shape(K)[-1], tf.float32)
            H = H / tf.math.log(tf.maximum(C, 2.0))

        return H

    def get_config(self):
        config = super().get_config()
        config.update({
            "alpha": self.alpha,
            "eps": self.eps,
            "normalize_by_logc": self.normalize_by_logc,
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)
@tf.keras.utils.register_keras_serializable()
class RenyiEntropyRegularizer(tf.keras.losses.Loss):
    def __init__(self, mode="min", name="renyi_entropy_regularizer", **kwargs):
        """
        mode = "min"  -> minimiza entropía
        mode = "max"  -> maximiza entropía
        """
        super().__init__(name=name, **kwargs)
        if mode not in ["min", "max"]:
            raise ValueError("mode debe ser 'min' o 'max'")
        self.mode = mode

    def call(self, y_true, y_pred):
        """
        y_pred: salida de entropía, típicamente (B, 1) o (B, F)
        """
        H = tf.cast(y_pred, tf.float32)
        H = tf.reduce_mean(H, axis=-1)   # (B,)

        if self.mode == "min":
            return H
        else:
            return -H

    def get_config(self):
        config = super().get_config()
        config.update({"mode": self.mode})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

def inception_block(x, kernel_sigma):
    """
    Versión de un solo kernel.
    """
    kernel = GaussianKernelLayer(name="gaussian_layer")(
        [x, tf.convert_to_tensor(kernel_sigma, dtype=tf.float32)]
    )


    return kernel

@tf.keras.utils.register_keras_serializable()
class NormalizedBinaryCrossentropy(tf.keras.losses.Loss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, y_true, y_pred):
        batch_size = tf.shape(y_pred)[0]

        cce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

        left = tf.tile(tf.expand_dims([1.0, 0.0], axis=0), [batch_size, 1])
        right = tf.tile(tf.expand_dims([0.0, 1.0], axis=0), [batch_size, 1])

        cce_left = tf.keras.losses.binary_crossentropy(left, y_pred)
        cce_right = tf.keras.losses.binary_crossentropy(right, y_pred)

        cce_norm = tf.divide(cce, (cce_left + cce_right))
        return cce_norm


class InspectableMultiHeadAttention(tf.keras.layers.MultiHeadAttention):
    def get_projection_weights(self):
        if not self.built:
            raise ValueError("La capa no ha sido construida.")

        weights_dict = {
            "query": self._query_dense.kernel.numpy(),
            "key": self._key_dense.kernel.numpy(),
            "value": self._value_dense.kernel.numpy(),
            "output": self._output_dense.kernel.numpy(),
        }
        return weights_dict


class InspectableTransformerEncoder(TransformerEncoder):
    def build(self, input_shape):
        hidden_dim = input_shape[-1]
        key_dim = int(hidden_dim // self.num_heads)

        self._self_attention_layer = InspectableMultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=key_dim,
            value_dim=key_dim,
            dropout=self.dropout,
            name="self_attention_inspectable",
        )

        self._self_attention_layer_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self._self_attention_dropout = tf.keras.layers.Dropout(rate=self.dropout)
        self._feedforward_intermediate_dense = tf.keras.layers.Dense(
            self.intermediate_dim, activation=self.activation
        )
        self._feedforward_output_dense = tf.keras.layers.Dense(hidden_dim)
        self._feedforward_layer_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self._feedforward_dropout = tf.keras.layers.Dropout(rate=self.dropout)

        self._last_attention_scores = None
        self.built = True

    def call(self, inputs, padding_mask=None, training=False):
        attention_output, attention_scores = self._self_attention_layer(
            query=inputs,
            key=inputs,
            value=inputs,
            attention_mask=padding_mask,
            training=training,
            return_attention_scores=True,
        )

        self._last_attention_scores = attention_scores

        attention_output = self._self_attention_dropout(attention_output, training=training)
        attention_output = self._self_attention_layer_norm(inputs + attention_output)

        ff_output = self._feedforward_intermediate_dense(attention_output)
        ff_output = self._feedforward_output_dense(ff_output)
        ff_output = self._feedforward_dropout(ff_output, training=training)
        output = self._feedforward_layer_norm(attention_output + ff_output)

        return output

    def get_attention_scores(self):
        if self._last_attention_scores is None:
            raise ValueError(
                "No se han calculado aún los attention scores. Haz un forward pass primero."
            )
        return self._last_attention_scores

    def get_attention_weights(self):
        if not self.built:
            raise ValueError("La capa Encoder no ha sido construida.")
        return self._self_attention_layer.get_projection_weights()

class DynamicSchedule(tf.keras.callbacks.Callback):
    def __init__(self, total_epochs, delta=10):
        super().__init__()
        self.total_epochs = total_epochs
        self.delta = delta
        self.lambda_val = 0.0

    def get_lambda(self, epoch):
        p = epoch / self.total_epochs
        return 2*(1 - np.exp(-self.delta * p)) / (1 + np.exp(-self.delta * p))

    def on_epoch_begin(self, epoch, logs=None):
        self.lambda_val = self.get_lambda(epoch)

        if hasattr(self.model, "loss_weights") and isinstance(self.model.loss_weights, dict):
            self.model.loss_weights["out_activation"] = 1.0
            self.model.loss_weights["kernel_entropy"] = self.lambda_val

In [5]:
# ============================================================
# DATASET BUILDER PARA TGARNet
# ============================================================
def make_tgarnet_ds_from_indices(X, y, idxs, training, batch_size, seed):
    """
    Devuelve un tf.data.Dataset compatible con la doble salida de TGARNet:
      - out_activation: etiquetas one-hot (B, 2)
      - kernel_entropy: dummy target (B, 1)
    """
    x = X[idxs].astype(np.float32)
    y_bin = y[idxs].astype(np.int32)

    y_out = tf.keras.utils.to_categorical(y_bin, num_classes=2).astype(np.float32)
    y_entropy = np.zeros((len(idxs), 1), dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices(
        (
            x,
            {
                "out_activation": y_out,
                "kernel_entropy": y_entropy,
            },
        )
    )

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [6]:
def make_ds_from_indices(X, y, groups, idxs, training, with_groups, batch_size, seed):
    x = X[idxs]
    yy = y[idxs]

    if with_groups:
        gg = groups[idxs].astype(str)
        ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [7]:
def TGARNet(
    nb_classes=2,
    Chans=19,
    Samples=256,
    norm_rate=0.25,
    alpha=2,
    num_heads=3,
    intermediate_dim=128,
    filters=3,
    dropoutRate=0.3,
    kernel_sigma=1.0,
):
    input1 = Input(shape=(Chans, Samples), name="input_eeg")

    # 1) Reorganize data for Transformer: (B, C, T) -> (B, T, C)
    x = Permute((2, 1), name="permute_to_tokens")(input1)

    # 2) Layer norm before Transformer
    x = LayerNormalization(name="pre_transformer_ln")(x)

    # 3) Transformer encoder
    transformer_encoder = InspectableTransformerEncoder(
        num_heads=num_heads,
        intermediate_dim=intermediate_dim,
        name="transformer_encoder"
    )
    x = transformer_encoder(x)

    # 4) Layer norm after Transformer
    x = LayerNormalization(name="post_transformer_ln")(x)

    # 5) Restore original shape: (B, T, C) -> (B, C, T, 1)
    x = Permute((2, 1), name="permute_back_ct")(x)
    x = Reshape((Chans, Samples, 1), name="reshape_for_kernel")(x)

    # 6) Single-kernel Gaussian connectivity block
    kernel_out = inception_block(x, kernel_sigma)   # (B, C, C, 1)

    # 7) Rényi entropy branch
    kernel_out_T = Permute((3, 1, 2), name="kernel_transpose_for_entropy")(kernel_out)  # (B, 1, C, C)
    kernel_entropy = RenyiEntropyLayer(
        alpha=alpha,
        normalize_by_logc=True,
        name="kernel_entropy"
    )(kernel_out_T)

    # 8) Extra convolutional stack over connectivity map
    final_conv = Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        activation="selu",
        name="Conv2D_1"
    )(kernel_out)
    final_conv = BatchNormalization(name="bn_1")(final_conv)

    final_conv = Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        activation="selu",
        name="Conv2D_2"
    )(final_conv)
    final_conv = BatchNormalization(name="bn_2")(final_conv)

    # 9) Classification head
    flat = Flatten(name="flatten")(final_conv)
    drop = Dropout(dropoutRate, name="dropout")(flat)

    softmax = Dense(
        nb_classes,
        activation="softmax",
        name="out_activation",
        kernel_constraint=max_norm(norm_rate)
    )(drop)

    model = Model(
        inputs=input1,
        outputs={
            "out_activation": softmax,
            "kernel_entropy": kernel_entropy
        },
        name="TGARNet"
    )
    return model


def suggest_model_args(trial, model_name, base_model_args):
    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Only 'tgarnet' is supported in this notebook.")

    return {
        **base_model_args,
        "filters": trial.suggest_int("filters", 2, 8),
        "kernel_sigma": trial.suggest_float("kernel_sigma", 1.0, 20.0),
        "num_heads": trial.suggest_int("num_heads", 1, 5),
        "intermediate_dim": trial.suggest_categorical("intermediate_dim", [16, 32, 64, 128]),
    }


def build_eeg_model(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Only 'tgarnet' is supported in this notebook.")

    nb_classes = kwargs.pop("nb_classes")
    Chans = kwargs.pop("Chans")
    Samples = kwargs.pop("Samples")

    return TGARNet(
        nb_classes=nb_classes,
        Chans=Chans,
        Samples=Samples,
        norm_rate=kwargs.get("norm_rate", 0.25),
        alpha=kwargs.get("alpha", 2),
        num_heads=kwargs["num_heads"],
        intermediate_dim=kwargs["intermediate_dim"],
        filters=kwargs["filters"],
        dropoutRate=kwargs.get("dropoutRate", 0.3),
        kernel_sigma=kwargs["kernel_sigma"],
    )


def suggest_compile_args(trial, model_name, base_compile_args=None):
    if base_compile_args is None:
        base_compile_args = {}

    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Only 'tgarnet' is supported in this notebook.")

    return {
        **base_compile_args,
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True),
        "schedule_delta": trial.suggest_float("schedule_delta", 1.0, 20.0),
    }


def build_compile_config(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "tgarnet":
        raise ValueError("Only 'tgarnet' is supported in this notebook.")

    optimizer = Adam(learning_rate=kwargs["learning_rate"])

    compile_args = {
        "optimizer": optimizer,
        "loss": {
            "out_activation": NormalizedBinaryCrossentropy(
                name="NormalizedBinaryCrossentropy"
            ),
            "kernel_entropy": RenyiEntropyRegularizer(
                name="RenyiEntropyRegularizer"
            ),
        },
        "loss_weights": {
            "out_activation": 0.5,
            "kernel_entropy": 0.5,
        },
        "metrics": {
            "out_activation": [
                tf.keras.metrics.BinaryAccuracy(name="acc"),
                tf.keras.metrics.AUC(name="auc"),
            ]
        }
    }

    callbacks = [
        DynamicSchedule(
            total_epochs=kwargs["total_epochs"],
            delta=kwargs["schedule_delta"],
        )
    ]

    return compile_args, callbacks

In [8]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================
MODEL_NAME = "tgarnet"

SEED_FOLDS = 123                  # fija las particiones
TRAINING_SEEDS = [42, 123, 456, 789, 2024, 7, 99, 314, 2718, 5000]

N_FOLDS = 5
BATCH_SIZE = 64
EPOCHS_FINAL = 100

EXCLUDED_SUBJECTS = {"v56p"}

DATA_ROOT = "/kaggle/input/datasets/javierceron/tdha-data"
DANNA_CSV = "/kaggle/input/datasets/alejandragomezr/results-by-fold-danna/final_results-Danna.csv"

RESULTS_ROOT = "/kaggle/working/results_tgarnet_10seeds"
PARTITIONS_DIR = os.path.join(RESULTS_ROOT, "partitions")
ALL_CURVES_DIR = os.path.join(RESULTS_ROOT, "training_curves")
ALL_MODELS_DIR = os.path.join(RESULTS_ROOT, "models")
ALL_CM_DIR = os.path.join(RESULTS_ROOT, "cm")
ALL_WEIGHTS_DIR = os.path.join(RESULTS_ROOT, "weights")

for d in [RESULTS_ROOT, PARTITIONS_DIR, ALL_CURVES_DIR, ALL_MODELS_DIR, ALL_CM_DIR, ALL_WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# UTILIDADES
# ============================================================
def set_all_seeds(seed: int):
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def summarize_subject_split(split_name, subj_names, subj_labels, max_show=60):
    n0 = int(np.sum(subj_labels == 0))
    n1 = int(np.sum(subj_labels == 1))
    print(f"{split_name}: n_subj={len(subj_names)} | Control={n0} | ADHD={n1}")

    pairs = [f"{n}({'C' if l == 0 else 'A'})" for n, l in zip(subj_names, subj_labels)]
    if len(pairs) <= max_show:
        print("  " + ", ".join(pairs))
    else:
        print("  " + ", ".join(pairs[:max_show]) + " ...")


def evaluate_subject_level(model, test_ds_xyg, threshold=0.5):
    subject_preds = defaultdict(list)
    subject_probs = defaultdict(list)
    subject_true = {}

    for xb, yb, sb in test_ds_xyg:
        raw_pred = model(xb, training=False)
        prob = extract_positive_prob(raw_pred)
        pred = (prob >= threshold).astype(int)

        y_np = yb.numpy().astype(int)
        s_np = sb.numpy()

        for name, p, pr, yt in zip(s_np, pred, prob, y_np):
            name = name.decode("utf-8") if isinstance(name, bytes) else str(name)
            subject_preds[name].append(int(p))
            subject_probs[name].append(float(pr))
            subject_true[name] = int(yt)

    y_true, y_pred, y_prob = [], [], []
    for subj in sorted(subject_preds.keys()):
        y_true.append(subject_true[subj])
        y_pred.append(int(np.bincount(subject_preds[subj]).argmax()))
        y_prob.append(float(np.mean(subject_probs[subj])))

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    if len(np.unique(y_true)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "accuracy": float(acc),
        "balanced_acc": float(bacc),
        "f1": float(f1),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "cm": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


# ============================================================
# CARGAR LOS MEJORES HPs DE DANNA DESDE CSV
# ============================================================
danna_df = pd.read_csv(DANNA_CSV)

danna_df["best_model_args"] = danna_df["best_model_args"].apply(ast.literal_eval)
danna_df["best_compile_args"] = danna_df["best_compile_args"].apply(ast.literal_eval)

DANNA_FIXED_BY_FOLD = {}
for _, row in danna_df.iterrows():
    fold_id = int(row["fold"])
    DANNA_FIXED_BY_FOLD[fold_id] = {
        "model_args": row["best_model_args"],
        "compile_args": row["best_compile_args"],
        "ref_metrics": {
            "accuracy": float(row["accuracy"]),
            "balanced_acc": float(row["balanced_acc"]),
            "f1": float(row["f1"]),
            "auc": float(row["auc"]),
        }
    }

print("HPs de Danna cargados por fold:")
for fid in sorted(DANNA_FIXED_BY_FOLD.keys()):
    print(f"  Fold {fid}: OK")


# ============================================================
# DATASET
# ============================================================
adhd_dir = os.path.join(DATA_ROOT, "ADHD")
control_dir = os.path.join(DATA_ROOT, "Control")

dataset_tf = EEGDataset_ADHD_TF(
    adhd_dir=adhd_dir,
    control_dir=control_dir,
    lowcut=0.5,
    highcut=60.0,
    notch=50.0,
    window=2.0,
    overlap=0.5,
    default_fs=128
)

# excluir sujetos problemáticos
n_before = len(dataset_tf.samples)
dataset_tf.samples = [
    sample for sample in dataset_tf.samples
    if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
]
n_after = len(dataset_tf.samples)

print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"Subjects removed: {n_before - n_after}")
print(f"Remaining subjects: {n_after}")

if len(dataset_tf.samples) == 0:
    raise ValueError("No subjects left after exclusion.")

X, y, groups = build_epoch_arrays(dataset_tf)
N_CH, N_T = int(X.shape[1]), int(X.shape[2])

print("Epoch arrays:")
print("  X:", X.shape, X.dtype)
print("  y:", y.shape, "labels:", np.unique(y))
print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))
print("Model input shape (C,T):", (N_CH, N_T))

subj_names, subj_labels = build_subject_table_from_epochs(groups, y)

print("Subject table:")
print("  n_subjects:", len(subj_names))
print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))


# ============================================================
# CONSTRUIR PARTICIONES FIJAS
# ============================================================
outer = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED_FOLDS
)

X_dummy = np.zeros((len(subj_names), 1), dtype=np.float32)

FIXED_SPLITS = []
partitions_rows = []

for fold_id, (trainval_subj_idx, test_subj_idx) in enumerate(
    outer.split(X_dummy, subj_labels, groups=subj_names)
):
    print(f"\n================ FIXED FOLD {fold_id+1}/{N_FOLDS} ================")

    inner = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=SEED_FOLDS + fold_id
    )

    inner_X_dummy = np.zeros((len(trainval_subj_idx), 1), dtype=np.float32)
    inner_y = subj_labels[trainval_subj_idx]
    inner_groups = subj_names[trainval_subj_idx]

    tr_rel, val_rel = next(inner.split(inner_X_dummy, inner_y, groups=inner_groups))

    train_subj_idx = trainval_subj_idx[tr_rel]
    val_subj_idx = trainval_subj_idx[val_rel]

    tr_names = subj_names[train_subj_idx]
    va_names = subj_names[val_subj_idx]
    te_names = subj_names[test_subj_idx]

    tr_labels = subj_labels[train_subj_idx]
    va_labels = subj_labels[val_subj_idx]
    te_labels = subj_labels[test_subj_idx]

    summarize_subject_split("Train subjects", tr_names, tr_labels)
    summarize_subject_split("Val subjects", va_names, va_labels)
    summarize_subject_split("Test subjects", te_names, te_labels)

    if len(np.unique(tr_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: train set does not contain both classes.")
    if len(np.unique(va_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: validation set does not contain both classes.")
    if len(np.unique(te_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: test set does not contain both classes.")

    tr_idx = epoch_indices_from_subjects(groups, tr_names)
    val_idx = epoch_indices_from_subjects(groups, va_names)
    te_idx = epoch_indices_from_subjects(groups, te_names)

    trainval_names = np.concatenate([tr_names, va_names])
    trainval_idx = epoch_indices_from_subjects(groups, trainval_names)

    FIXED_SPLITS.append({
        "fold_id": fold_id,
        "tr_names": tr_names,
        "va_names": va_names,
        "te_names": te_names,
        "tr_labels": tr_labels,
        "va_labels": va_labels,
        "te_labels": te_labels,
        "tr_idx": tr_idx,
        "val_idx": val_idx,
        "te_idx": te_idx,
        "trainval_idx": trainval_idx,
    })

    partitions_rows.append({
        "fold": fold_id,
        "train_subjects": list(tr_names),
        "val_subjects": list(va_names),
        "test_subjects": list(te_names),
        "n_train_subjects": len(tr_names),
        "n_val_subjects": len(va_names),
        "n_test_subjects": len(te_names),
        "n_train_epochs": len(tr_idx),
        "n_val_epochs": len(val_idx),
        "n_test_epochs": len(te_idx),
    })

df_partitions = pd.DataFrame(partitions_rows)
df_partitions.to_csv(os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"), index=False)
print("\nParticiones fijas guardadas en:", os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))


# ============================================================
# CORRER UNA SEED
# ============================================================
def run_tgarnet_one_seed(training_seed, save_artifacts=True):
    print(f"\n{'='*90}")
    print(f"RUN TGARNet | training_seed = {training_seed}")
    print(f"{'='*90}")

    seed_root = os.path.join(RESULTS_ROOT, f"seed_{training_seed}")
    curves_dir = os.path.join(seed_root, "training_curves")
    models_dir = os.path.join(seed_root, "models")
    cm_dir = os.path.join(seed_root, "cm")
    weights_dir = os.path.join(seed_root, "weights")

    for d in [seed_root, curves_dir, models_dir, cm_dir, weights_dir]:
        os.makedirs(d, exist_ok=True)

    fold_results = []
    fold_histories = {}
    fold_cms = {}
    super_cm = np.zeros((2, 2), dtype=int)

    for split in FIXED_SPLITS:
        fold_id = split["fold_id"]
        print(f"\n---------------- Fold {fold_id+1}/{N_FOLDS} | seed={training_seed} ----------------")

        fold_seed = int(training_seed * 100 + fold_id)

        cfg = DANNA_FIXED_BY_FOLD[fold_id]

        best_model_args = dict(cfg["model_args"])
        best_compile_args = dict(cfg["compile_args"])

        # asegurar compatibilidad con los datos actuales
        best_model_args["Chans"] = N_CH
        best_model_args["Samples"] = N_T
        best_compile_args["total_epochs"] = EPOCHS_FINAL

        trainval_idx = split["trainval_idx"]
        te_idx = split["te_idx"]

        # dataset de entrenamiento final (train + val)
        trainval_ds_tg = make_tgarnet_ds_from_indices(
            X, y, trainval_idx,
            training=True,
            batch_size=BATCH_SIZE,
            seed=fold_seed
        )

        # dataset de test para evaluación por sujeto
        test_ds_xyg = make_ds_from_indices(
            X, y, groups, te_idx,
            training=False,
            with_groups=True,
            batch_size=BATCH_SIZE,
            seed=fold_seed
        )

        # modelo desde cero
        set_all_seeds(fold_seed)

        model = build_eeg_model(MODEL_NAME, **best_model_args)

        compile_args, final_extra_callbacks = build_compile_config(
            MODEL_NAME,
            **best_compile_args
        )
        model.compile(**compile_args)

        hist = model.fit(
            trainval_ds_tg,
            epochs=EPOCHS_FINAL,
            verbose=1,
            callbacks=final_extra_callbacks,
        )

        fold_histories[fold_id] = hist.history

        if save_artifacts:
            model.save(os.path.join(models_dir, f"fold_{fold_id}.keras"))
            model.save_weights(os.path.join(weights_dir, f"fold_{fold_id}.weights.h5"))

            plot_fold_training_curve(
                hist.history,
                fold_id,
                save_path=os.path.join(curves_dir, f"fold_{fold_id}_curves.png"),
                metrics=("loss", "out_activation_acc", "out_activation_auc"),
            )

        fold_eval = evaluate_subject_level(model, test_ds_xyg, threshold=0.5)
        cm = fold_eval["cm"]

        super_cm += cm
        fold_cms[fold_id] = cm

        if save_artifacts:
            plot_fold_confusion_matrix(
                cm,
                fold_id,
                save_path=os.path.join(cm_dir, f"fold_{fold_id}_cm.png"),
            )

        row = {
            "seed": training_seed,
            "fold": fold_id,
            "accuracy": fold_eval["accuracy"],
            "balanced_acc": fold_eval["balanced_acc"],
            "f1": fold_eval["f1"],
            "auc": fold_eval["auc"],
            "best_model_args": dict(best_model_args),
            "best_compile_args": dict(best_compile_args),
        }
        fold_results.append(row)

        print(
            f"Fold {fold_id} | "
            f"Acc={row['accuracy']:.3f} | "
            f"BAcc={row['balanced_acc']:.3f} | "
            f"F1={row['f1']:.3f} | "
            f"AUC={row['auc']:.3f}"
        )

        del model
        gc.collect()
        tf.keras.backend.clear_session()

    if save_artifacts:
        plot_all_training_curves(
            fold_histories,
            curves_dir,
            metrics=("loss", "out_activation_acc", "out_activation_auc"),
        )
        plot_all_confusion_matrices(fold_cms, super_cm, cm_dir)

    df_seed = pd.DataFrame(fold_results)

    seed_summary = {
        "seed": training_seed,
        "mean_accuracy": float(df_seed["accuracy"].mean()),
        "std_accuracy": float(df_seed["accuracy"].std(ddof=1)),
        "mean_balanced_acc": float(df_seed["balanced_acc"].mean()),
        "std_balanced_acc": float(df_seed["balanced_acc"].std(ddof=1)),
        "mean_f1": float(df_seed["f1"].mean()),
        "std_f1": float(df_seed["f1"].std(ddof=1)),
        "mean_auc": float(df_seed["auc"].mean()),
        "std_auc": float(df_seed["auc"].std(ddof=1)),
    }

    df_seed.to_csv(os.path.join(seed_root, "fold_results.csv"), index=False)
    pd.DataFrame([seed_summary]).to_csv(os.path.join(seed_root, "seed_summary.csv"), index=False)

    print("\nResumen seed:")
    print(pd.DataFrame([seed_summary]).T)

    return df_seed, seed_summary


# ============================================================
# CORRER LAS 10 SEMILLAS
# ============================================================
all_fold_results = []
all_seed_summaries = []

for training_seed in TRAINING_SEEDS:
    df_seed, seed_summary = run_tgarnet_one_seed(
        training_seed=training_seed,
        save_artifacts=True
    )
    all_fold_results.append(df_seed)
    all_seed_summaries.append(seed_summary)

df_all_folds = pd.concat(all_fold_results, axis=0, ignore_index=True)
df_all_seeds = pd.DataFrame(all_seed_summaries)

df_all_folds.to_csv(os.path.join(RESULTS_ROOT, "all_fold_results.csv"), index=False)
df_all_seeds.to_csv(os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"), index=False)

print("\nRESULTADOS POR SEED")
print(df_all_seeds)

print("\nRESUMEN GLOBAL")
print(df_all_seeds[["mean_accuracy", "mean_balanced_acc", "mean_f1", "mean_auc"]].agg(["mean", "std"]))

print("\nArchivos principales guardados en:")
print(" -", os.path.join(RESULTS_ROOT, "all_fold_results.csv"))
print(" -", os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"))
print(" -", os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))

HPs de Danna cargados por fold:
  Fold 0: OK
  Fold 1: OK
  Fold 2: OK
  Fold 3: OK
  Fold 4: OK
Excluded subjects: ['v56p']
Subjects removed: 1
Remaining subjects: 120
Epoch arrays:
  X: (16640, 19, 256) float32
  y: (16640,) labels: [0 1]
  groups: (16640,) n_subjects: 120
Model input shape (C,T): (19, 256)
Subject table:
  n_subjects: 120
  Control: 59 | ADHD: 61

================ FIXED FOLD 1/5 ================
Train subjects: n_subj=72 | Control=35 | ADHD=37
  v109.mat(C), v10p.mat(A), v110.mat(C), v113.mat(C), v114.mat(C), v115.mat(C), v117.mat(C), v120.mat(C), v121.mat(C), v127.mat(C), v129.mat(C), v12p.mat(A), v131.mat(C), v138.mat(C), v140.mat(C), v147.mat(C), v149.mat(C), v151.mat(C), v173.mat(A), v179.mat(A), v181.mat(A), v183.mat(A), v196.mat(A), v198.mat(A), v1p.mat(A), v204.mat(A), v206.mat(A), v20p.mat(A), v213.mat(A), v219.mat(A), v22p.mat(A), v231.mat(A), v234.mat(A), v238.mat(A), v246.mat(A), v24p.mat(A), v25p.mat(A), v263.mat(A), v265.mat(A), v274.mat(A), v27p.mat(A)

I0000 00:00:1776969768.262782      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776969768.268886      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/100


I0000 00:00:1776969777.935948      70 service.cc:152] XLA service 0x7aa8240091a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776969777.935985      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776969777.935989      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776969779.111635      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


 15/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - kernel_entropy_loss: 0.9895 - loss: 0.7339 - out_activation_acc: 0.5549 - out_activation_auc: 0.5521 - out_activation_loss: 0.4784

I0000 00:00:1776969785.791966      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


204/204 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - kernel_entropy_loss: 0.9879 - loss: 0.7211 - out_activation_acc: 0.5632 - out_activation_auc: 0.5820 - out_activation_loss: 0.4542
Epoch 2/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - kernel_entropy_loss: 0.9772 - loss: 0.7029 - out_activation_acc: 0.5723 - out_activation_auc: 0.6544 - out_activation_loss: 0.4286
Epoch 3/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - kernel_entropy_loss: 0.9552 - loss: 0.6538 - out_activation_acc: 0.7075 - out_activation_auc: 0.7850 - out_activation_loss: 0.3525
Epoch 4/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - kernel_entropy_loss: 0.9210 - loss: 0.6171 - out_activation_acc: 0.7602 - out_activation_auc: 0.8245 - out_activation_loss: 0.3132
Epoch 5/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - kernel_entropy_loss: 0.8604 - loss: 0.5793 - out_activation_acc: 0.7753 - out_activation_auc: 0.8320 - out_activation_loss: 0.2982
Epoch 6/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - kernel_entropy_loss: 0.74